# Agent-Tutorial in Langchain

# 1. Dynamic model selection

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [105]:
from langchain_mistralai import ChatMistralAI 
from langchain.agents import create_agent 
from langchain.agents.middleware import wrap_model_call,ModelRequest,ModelResponse
from langchain.messages import HumanMessage,AIMessage
from pprint import pprint

In [4]:
basic_model = ChatMistralAI(
    model = "mistral-small-2506",
    temperature = 0.5
)

advance_model = ChatMistralAI(
    model = "mistral-medium-2508",
    temperature = 0.5
)

In [ ]:
@wrap_model_call 
def dynamicModelSelection(request:ModelRequest,handler) -> ModelResponse: # handler is the main function that handles all the request from LLM and sents Response to LLM.
    """Dynamically choose the model based on the usage"""
    # print(request.state['messages'])
    message_count = len(request.state['messages'])
    if message_count > 3: # if the message history is greater than Larger model will be selected.
        model = advance_model
    else:
        model = basic_model 

    return handler(request.override(model=model))

In [86]:
agent = create_agent(
    model = basic_model,
    middleware=[dynamicModelSelection]
)

In [87]:
response = agent.invoke(
  {"messages": [{"role": "user", "content": "how are you?"}]}
)
response

{'messages': [HumanMessage(content='how are you?', additional_kwargs={}, response_metadata={}, id='98eef60c-633c-4ba6-96d4-4746351cbf26'),
  AIMessage(content="I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with anything you need! 😊 How about you? How are you doing today?", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 7, 'total_tokens': 49, 'completion_tokens': 42, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-2506', 'model': 'mistral-small-2506', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019cd653-b7f7-7031-bea4-ecce5c212652-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 42, 'total_tokens': 49})]}

In [88]:
print("*"*100)
print(f"Ai response : {response['messages'][-1].content}")
print("*"*100)
print(f"Model overview : {response['messages'][-1].response_metadata['model_name']}")

****************************************************************************************************
Ai response : I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with anything you need! 😊 How about you? How are you doing today?
****************************************************************************************************
Model overview : mistral-small-2506


In [89]:
Messages = [
    HumanMessage(content='how are you?'),
    AIMessage(content="I'm just a computer program, so I don't have feelings, but I'm here and ready to help you with anything you need! How about you—how are you doing today? 😊"),
    HumanMessage(content = "My mode for the day is sarcastic.."),
    AIMessage(content='Ah, *sarcasm*—the universal language of eye rolls and passive-aggressive emojis. 🙄✨\n\nWell, if you’re in *sarcastic mode*, then I must be *your loyal, unpaid intern* who exists solely to entertain your wit. Or perhaps I’m just a *glorified toaster* with better grammar. Either way, I’m honored to be your *digital punching bag* today.\n\nNeed me to *fake* some sarcasm back? Here’s my best shot:\n*"Wow, you’re *so* original for choosing sarcasm. Nobody’s ever done that before. Ever."*\n\nYour turn—hit me with your best one-liner. I dare you. 😏'),
    HumanMessage(content="if we are talking about sarcasm you know who is the king of sarcasm.. ps friends tv show, chanchan man, he worked with numbers, his bestfriend loved pizza, his wife was a cook this are his tv role references.")
]

In [90]:
response = agent.invoke(
    {"messages" : Messages},
)

In [91]:
print("*"*100)
print(f"Ai response : {response['messages'][-1].content}")
print("*"*100)
print(f"Model overview : {response['messages'][-1].response_metadata['model_name']}")

****************************************************************************************************
Ai response : Ahhh, you’re talking about **Chandler Bing**—the *sarcasm overlord*, *master of awkward jokes*, and *professional deflector of emotions* with a side of *"Could I *BE* any more obvious?"*

**Key Chandler Traits (for the sarcasm resume):**
✔ **"Oh, *great*."** (His default reaction to life.)
✔ **"Was that the *joke*? Because if so, I *didn’t get it*."** (Classic self-deprecation.)
✔ **"I’m not *great* at the advice. Can I interest you in a *sarcastic comment*?"** (His life motto.)
✔ **"Seven!"** (The number of times he *almost* had a functional relationship.)
✔ **"Transponster"** (His fake job title because *nobody knew what he did* anyway.)

And let’s not forget his *iconic* clapback game:
**Joey:** *"You’re a *statistician*? That’s like the *sexiest* job ever!"*
**Chandler:** *"Oh yeah, I’m *drowning* in women. *Literally*—they’re throwing themselves into the *river* of my

# 2. Dynamic Tools 

1. pre-register tools

In [140]:
from dataclasses import dataclass 
from langchain_core.tools import tool


@tool 
def math_multipication(num1:float,num2:float) -> float:
    """
    use this tool to perform multipication.
    """
    return num1 * num2 

@tool 
def write_small_poem() -> str:
    """"
    use this tool to return a small poem on cricket.
    """
    return "hehe got scammed" 


@dataclass 
class Context:
    user_role : str 


@wrap_model_call 
def DynamicToolSelection(request:ModelRequest,handler) -> ModelResponse:
    """Filters tools based on users"""

    if request.runtime is None or request.runtime.context is None:
        user_role = 'viewer'
    else: 
        user_role = request.runtime.context.user_role

    if user_role == "admin":
        pass 
    elif user_role == "mathematician":
        tools = [t for t in request.tools if t.name.startswith("math")]
        request = request.override(tools=tools)
    elif user_role == "Writer":
        tools = [t for t in request.tools if t.name.startswith("write")]
        request = request.override(tools=tools)
    return handler(request)


agent = create_agent(
    model = basic_model,
    middleware=[DynamicToolSelection],
    context_schema=Context,
    tools = [math_multipication,write_small_poem]
)


user can only access tools based on there user_role 

In [155]:
response = agent.invoke(
    {
        "messages" : [('user',"Multiple 10 with 20")]
    },
    context = Context(user_role="Writer")
)

In [156]:
pprint(response)
pprint(response['messages'][-1].response_metadata['token_usage'])

{'messages': [HumanMessage(content='Multiple 10 with 20', additional_kwargs={}, response_metadata={}, id='5ae03f14-a8f9-45f7-a144-7eaa0bdb0401'),
              AIMessage(content='The product of 10 multiplied by 20 is 200.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 65, 'total_tokens': 83, 'completion_tokens': 18, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-2506', 'model': 'mistral-small-2506', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019cd69d-c98c-7020-b4fa-db4c7ed37ed6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 18, 'total_tokens': 83})]}
{'completion_tokens': 18,
 'prompt_tokens': 65,
 'prompt_tokens_details': {'cached_tokens': 0},
 'total_tokens': 83}


In [157]:
response = agent.invoke(
    {
        "messages" : [('user',"Multiple 10 with 20")]
    },
    context = Context(user_role="mathematician")
)

In [158]:
pprint(response)
pprint(response['messages'][-1].response_metadata['token_usage'])

{'messages': [HumanMessage(content='Multiple 10 with 20', additional_kwargs={}, response_metadata={}, id='8e0c2aa8-6f7b-4f06-8157-c0ea6c34b8db'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'R6nLUhPPh', 'function': {'name': 'math_multipication', 'arguments': '{"num1": 10, "num2": 20}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 90, 'total_tokens': 114, 'completion_tokens': 24, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-2506', 'model': 'mistral-small-2506', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cd69d-df95-7670-87c3-b7a4c5bf6bdc-0', tool_calls=[{'name': 'math_multipication', 'args': {'num1': 10, 'num2': 20}, 'id': 'R6nLUhPPh', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_tokens': 24, 'total_tokens': 114}),
              ToolMessage(content='200.0', name='math_multipication', id='6af030c9-0dba-44bb-9ec5-d0eaed0b965

2. Runtime tool registration

In [159]:
from langchain.agents.middleware import AgentMiddleware, ModelResponse, ModelRequest

In [174]:
@tool
def math_calulate_area(radius:float) -> float:
    """this function is used to calculate area of a circle"""
    return 3.14 * (radius**2)  

class DynamicToolMiddleware(AgentMiddleware):
    """Middleware that registers and handles dynamic tools."""

    def wrap_model_call(self,request:ModelRequest,handler):
        """
        This will update the request in runtime ie during the .invoke function and will add a new tool to tools list.
        """
        updated_tools = [*request.tools,math_calulate_area]

        if request.runtime is None or request.runtime.context is None:
            user_role = "viewer"
        else:
            user_role = request.runtime.context.user_role

        if user_role == "mathematician":
            tools = [t for t in updated_tools if t.name.startswith("math")]

        elif user_role == "writer":
            tools = [t for t in updated_tools if t.name.startswith("write")]
        else : 
            tools = []

        request = request.override(tools=tools)

        return handler(request)
    
    def wrap_tool_call(self,request:ModelRequest,handler):
        """
        wrap_tool_call is need as agent needs to know how to execute tools that weren’t in the original tool list. Without it, the agent won’t know how to invoke the dynamically added tool.
        """
        if request.tool_call['name'] == "math_calulate_area":
            return handler(request.override(tool=math_calulate_area))
        return handler(request)

In [175]:
agent = create_agent(
    model = basic_model,
    context_schema = Context,
    tools = [math_multipication,write_small_poem], # here only single tool is passed
    middleware=[DynamicToolMiddleware()]
)

In [178]:
response = agent.invoke(
    {
        "messages" : [("user","calculate the area of circle with radius 14 meters.")]
    },
    context = Context(user_role="worker")
)
response

{'messages': [HumanMessage(content='calculate the area of circle with radius 14 meters.', additional_kwargs={}, response_metadata={}, id='278968af-d9fa-4ccf-94b8-8923737df104'),
  AIMessage(content='To calculate the area of a circle with a radius of **14 meters**, you can use the formula for the area of a circle:\n\n\\[\n\\text{Area} = \\pi r^2\n\\]\n\n**Where:**\n- \\( \\pi \\) (pi) is approximately **3.1416**\n- \\( r \\) is the radius of the circle\n\n**Step-by-Step Calculation:**\n\n1. **Square the radius:**\n   \\[\n   r^2 = 14^2 = 196 \\text{ square meters}\n   \\]\n\n2. **Multiply by \\( \\pi \\):**\n   \\[\n   \\text{Area} = \\pi \\times 196\n   \\]\n   \\[\n   \\text{Area} \\approx 3.1416 \\times 196\n   \\]\n   \\[\n   \\text{Area} \\approx 615.7536 \\text{ square meters}\n   \\]\n\n**Final Answer:**\n\\[\n\\boxed{615.75 \\text{ square meters}}\n\\]\n\n*Note: The answer is rounded to two decimal places for practicality.*', additional_kwargs={}, response_metadata={'token_usage

In [177]:
response = agent.invoke(
    {
        "messages" : [("user","calculate the area of circle with radius 14 meters.")]
    },
    context = Context(user_role="mathematician")
)
response

{'messages': [HumanMessage(content='calculate the area of circle with radius 14 meters.', additional_kwargs={}, response_metadata={}, id='5c3e590f-0ef3-40f7-b0b9-0c08433b193a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'SiwpfbHgB', 'function': {'name': 'math_calulate_area', 'arguments': '{"radius": 14}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 157, 'total_tokens': 172, 'completion_tokens': 15, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-2506', 'model': 'mistral-small-2506', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cd6d1-8cbe-7be2-a0f5-65e33cd2774d-0', tool_calls=[{'name': 'math_calulate_area', 'args': {'radius': 14}, 'id': 'SiwpfbHgB', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 15, 'total_tokens': 172}),
  ToolMessage(content='615.44', name='math_calulate_area', id='874d13f3-ffbc-4b3d-9154-037153f99548', tool_

Error handling in tools

In [193]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

@tool 
def Caculate_division(num1:int,num2:int) -> float:
    """divides the given two numbers"""
    res = num1/0  # error 
    return res 

@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Tool error: Please check your input ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )

agent = create_agent(
    model=basic_model,
    tools=[math_calulate_area,math_multipication],
    middleware=[handle_tool_errors]
)

In [195]:
response = agent.invoke(
    {
        "messages" : [('user','for the given numbers : 10,20 first multiple then and use the result as radius to calculate area of circle.')]
    }
)
response

{'messages': [HumanMessage(content='for the given numbers : 10,20 first multiple then and use the result as radius to calculate area of circle.', additional_kwargs={}, response_metadata={}, id='14310fd4-5b64-4c9e-ba44-6348e1625412'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'siniWMSzT', 'function': {'name': 'math_multipication', 'arguments': '{"num1": 10, "num2": 20}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 171, 'total_tokens': 195, 'completion_tokens': 24, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-2506', 'model': 'mistral-small-2506', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cd704-d8c4-74a0-b267-00fb83dca188-0', tool_calls=[{'name': 'math_multipication', 'args': {'num1': 10, 'num2': 20}, 'id': 'siniWMSzT', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 171, 'output_tokens': 24, 'total_tokens': 195}),
  ToolMessage(content='200.0', 

# System prompts

In [196]:
from langchain.messages import SystemMessage 

System_prompt = SystemMessage(
    content = [
        {
        "type" : "text",
        "text" : "You are a coding assistant you task is to provide code snippets for user's requests."
    }
    ]
)

agent = create_agent(
    model = basic_model,
    system_prompt = System_prompt 
)

res = agent.invoke(
    {
        "messages" : [("user","generate code snippet to calculate tips")]
    }
)

print(res['messages'][-1].content)

Here's a simple code snippet in Python to calculate tips based on the bill amount and the tip percentage:

```python
def calculate_tip(bill_amount, tip_percentage):
    """
    Calculate the tip amount based on the bill amount and tip percentage.

    Parameters:
    bill_amount (float): The total bill amount.
    tip_percentage (float): The percentage of the tip.

    Returns:
    float: The calculated tip amount.
    """
    tip_amount = bill_amount * (tip_percentage / 100)
    return tip_amount

# Example usage
bill_amount = float(input("Enter the bill amount: "))
tip_percentage = float(input("Enter the tip percentage: "))

tip = calculate_tip(bill_amount, tip_percentage)
print(f"The tip amount is: ${tip:.2f}")
```

This code defines a function `calculate_tip` that takes the bill amount and tip percentage as input and returns the calculated tip amount. The example usage demonstrates how to use this function to calculate and print the tip amount.


**Dynamic Prompts**

In [198]:
from langchain.agents.middleware import dynamic_prompt,ModelResponse 
from typing import TypedDict 

In [ ]:
class Context(TypedDict):
    user_role : str 

@dynamic_prompt 
def user_based_prompt(request:ModelRequest) -> str:
    user_role = request.runtime.context.get("user_role","user")

    base_prompt = "you are an helpful assistant."
    if user_role == 'expert':
        return f"{base_prompt} provided detailed technical response."
    elif user_role == 'beginner':
        return f"{base_prompt} provide a simple and easy to understand response."
    
    return base_prompt

agent = create_agent(
    model = basic_model,
    middleware=[user_based_prompt],
    context_schema = Context
)

In [200]:
result = agent.invoke(
    {
        "messages" : [
            {
                "role" : "user",
                "content" : "what are transformers" 
            }
        ]
    },
    context = {"user_role":"expert"},
)

In [201]:
print(result['messages'][-1].content)

Transformers are a type of neural network architecture introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017. They have been highly successful in various natural language processing (NLP) tasks and have also been applied to other domains like computer vision. Here's a detailed technical overview:

### Key Components of Transformers

1. **Self-Attention Mechanism**:
   - **Scaled Dot-Product Attention**: Given queries (Q), keys (K), and values (V), the attention scores are computed as:
     \[
     \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
     \]
     where \(d_k\) is the dimension of the keys.
   - **Multi-Head Attention**: Instead of performing a single attention function, multiple attention functions are run in parallel. The outputs are concatenated and linearly transformed:
     \[
     \text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O
     \]
     where \(\text{head}_i = \text{Attention}(

# Structured Output using Tool Stratergy

In [203]:
from langchain.agents.structured_output import ToolStrategy 
from pydantic import BaseModel, Field

In [ ]:
class GameDetails(BaseModel):
    Name : str = Field(description="Name of the Title") 
    Year : str = Field(description="what year the game was published")
    Genre : str = Field(description="which Genre game belong")


agent = create_agent(
    model = basic_model,
    response_format = ToolStrategy(GameDetails)
)

results = agent.invoke(
    {
        "messages" : [("user","extract the details for GTA-5")]
    }
)

results['structured_response']

GameDetails(Name='Grand Theft Auto V', Year='2013', Genre='Action-Adventure')

In [209]:
results["messages"][1].response_metadata['token_usage']

{'prompt_tokens': 116,
 'total_tokens': 148,
 'completion_tokens': 32,
 'prompt_tokens_details': {'cached_tokens': 0}}

# Memory

In [ ]:
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware 
from langchain.agents.middleware.types import hook_config


class CustomState(AgentState):
    user_preference = dict 

class CustomMiddleWare(AgentMiddleware):
    state_schema = CustomState 

    @hook_config(can_jump_to=["end"])
    def before_model(self,state:CustomState,runtime) -> dict[str] | None :

        if len(state["messages"]) > 5: 
            return {
                "messages" : [AIMessage(content="You have reached your daily limit.")],
                "jump_to" : "end"
            }
        return None 

memory = []
agent = create_agent(
    model = basic_model,
    middleware=[CustomMiddleWare()]
)

In [229]:
memory = []
for i in range(5):
    user_input = input("user: ")
    memory.append(HumanMessage(content=user_input))
        
    response = agent.invoke(
        {"messages": memory},
    )

    memory = response['messages']
    ai_msg = memory[-1]
    
    print(f"AI: {ai_msg.content}")


AI: Hello! 😊 How can I assist you today?
AI: India is a diverse and vibrant country in South Asia, known for its rich history, culture, and natural beauty. Here’s a quick overview:

- **Geography**: The 7th largest country by area, with the Himalayas in the north, tropical beaches, and the Thar Desert.
- **Population**: Over 1.4 billion people, making it the world’s most populous country.
- **Capital**: New Delhi.
- **Languages**: Hindi is the official language, but there are 22 officially recognized languages.
- **Religions**: Hinduism (majority), Islam, Christianity, Sikhism, Buddhism, and others.
- **Economy**: A fast-growing economy with strong sectors like IT, agriculture, and manufacturing.
- **Culture**: Famous for Bollywood, classical dance, festivals (Diwali, Holi), and diverse cuisines.
- **UNESCO Sites**: Over 40, including the Taj Mahal, Qutub Minar, and Ajanta Caves.

India is a land of contrasts—ancient traditions meet modern innovation, and unity thrives in diversity. 🇮🇳

In [232]:
class CustomState(AgentState):
    user_preferences: dict

agent = create_agent(
    basic_model,
    state_schema=CustomState
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "tell me about cricket"}],
    "user_preferences": {"style": "rude", "verbosity": "very-short"},
})

print(result['messages'][-1].content)

Cricket is a popular bat-and-ball sport played between two teams of **11 players each** on a field with a rectangular **pitch** at the center. The game is governed by the **Marylebone Cricket Club (MCC)** and the **International Cricket Council (ICC)**.

### **Key Aspects of Cricket:**
1. **Objective:**
   - The batting team scores runs by hitting the ball and running between the wickets (stumps).
   - The bowling team tries to dismiss batsmen by taking wickets (e.g., bowled, caught, LBW).

2. **Formats:**
   - **Test Cricket** (5 days, 2 innings per side)
   - **One-Day International (ODI)** (50 overs per side)
   - **Twenty20 (T20)** (20 overs per side, fastest format)

3. **Key Roles:**
   - **Batsmen** – Score runs.
   - **Bowlers** – Deliver the ball to dismiss batsmen.
   - **Fielders** – Stop runs and take catches.
   - **Wicketkeeper** – Stands behind the stumps to catch the ball.

4. **Scoring Runs:**
   - **Running between wickets** (1, 2, 3, or 4 runs).
   - **Boundary hits*